# Compute, use, and uncompute

Goal: apply a phase proportional to x*x+y without returning the computed value. A temporary register is quantum storage and cannot simply be discarded. `within` computes the temporary, `apply` uses it, and the compiler reverses the computation. `score` belongs only to this region; it is not an output.

In [ ]:
import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import Output, QNum, allocate, apply, hadamard_transform, phase, qfunc, within

@qfunc
def cost_phase(x: Output[QNum], y: Output[QNum]) -> None:
    allocate(2, x)
    allocate(2, y)
    hadamard_transform(x)
    hadamard_transform(y)
    with within():
        score: QNum[4] = x*x + y
    with apply():
        phase(score, 0.3)


In [ ]:
program = synth.synthesize(cost_phase, max_qubits=24, memory_limit_bytes=64*1024**2)
result = fq.run(program)
outcomes = dict(program.probabilities(result, registers=("x", "y")))
print(outcomes)
print(program.report["resources"])
print("Returned registers:", tuple(program.registers))


## Interpret the output

Probabilities remain 1/16. The amplitude of `(x,y)` has acquired exp(0.3j*(x*x+y)); this matters when composing a later interference step. Four unsigned bits can represent the maximum score 12. Only x and y are returned, and all temporary qubits must return to zero. The acceptance test checks the complex phases and temporary amplitudes, not just these probabilities.

Try another finite phase coefficient. You still do not need to design or reverse a multiplication circuit.